In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

BASE = Path("../Datasets/Raw_Data/Unlabeled_Data_With_GPS/Europe")

# Collect GPS files grouped by location name
records = {}
for gps_file in sorted(BASE.rglob("*GPS*.csv")):
    location = gps_file.stem.replace("_GPSData", "")
    df = pd.read_csv(gps_file, usecols=["attr_lat", "attr_lng"])
    records.setdefault(location, []).append(df)

location_data = {loc: pd.concat(dfs, ignore_index=True) for loc, dfs in records.items()}

COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990",
]
locations = sorted(location_data.keys())
color_map = {loc: COLORS[i % len(COLORS)] for i, loc in enumerate(locations)}

# Centre map on mean of all points
all_lats = pd.concat([d["attr_lat"] for d in location_data.values()])
all_lngs = pd.concat([d["attr_lng"] for d in location_data.values()])
m = folium.Map(location=[all_lats.mean(), all_lngs.mean()], zoom_start=5, tiles="CartoDB positron")

for loc, df in location_data.items():
    # Build GeoJSON FeatureCollection in one shot (no iterrows)
    features = [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [row.attr_lng, row.attr_lat]},
            "properties": {"tooltip": loc.replace("_", " ")},
        }
        for row in df.itertuples(index=False)
    ]
    geojson = {"type": "FeatureCollection", "features": features}

    color = color_map[loc]
    fg = folium.FeatureGroup(name=loc.replace("_", " "), show=True)
    folium.GeoJson(
        geojson,
        tooltip=folium.GeoJsonTooltip(fields=["tooltip"], aliases=["Location:"]),
        marker=folium.CircleMarker(
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=0,
        ),
    ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print(f"{len(locations)} locations, {sum(len(d) for d in location_data.values())} GPS points")
m


In [2]:
m.save("europe_gps_map.html")